In [28]:
import tskit
import pandas as pd
import allel
import numpy as np
import pickle
from collections import Counter
from collections import defaultdict
import matplotlib.pyplot as plt

chromosome = 15

In [29]:
# Read the tree sequence

ts = tskit.load("All_snps_with_outgroup_info_newAA/Chr" + str(chromosome) + ".trees")
ts_remapped = tskit.load("Remapped/Chr" + str(chromosome) + ".trees")
sample = pd.read_table("All_snps_with_outgroup_info_newAA/Chr" + str(chromosome) + "_relate.sample")
sample_remapped = pd.read_table("Remapped/Chr" + str(chromosome) + "_remapped.sample")

In [30]:
meta = pd.read_csv("TsinferMetaData_Vcf.txt")
meta.head()

sampleMeta = sample.merge(meta, how = "left", left_on = "ID_1", right_on = "ID")
sampleMeta = sampleMeta.iloc[1: , :]

complex_schema = tskit.MetadataSchema({
    'codec': 'json',
    'additionalProperties': False,
    'properties': {'name': {'description': 'ENA accession number',
                                 'type': 'string'},
                   'pop': {'description': 'Lineage',
                           'type': 'string'},
                    'subpop': {'description': 'Subspecies',
                           'type': 'string'}},
    'required': ['name', 'pop', 'subpop'],
    'type': 'object',
})


# Add meta data
tables = ts.dump_tables()
tablesRemapped = ts_remapped.dump_tables()
tables.individuals.metadata_schema = complex_schema
tablesRemapped.individuals.metadata_schema = complex_schema


# Add metadatats
metadata_column = [{"name": list(sampleMeta.ID_1)[ind], "pop": list(sampleMeta.Pop)[ind], "subpop": list(sampleMeta.SubPop)[ind]} for ind in range(ts.num_individuals)]
encoded_metadata_column = [
    tables.individuals.metadata_schema.validate_and_encode_row(r) for r in metadata_column
]
encoded_metadata_column_Remapped = [
    tablesRemapped.individuals.metadata_schema.validate_and_encode_row(r) for r in metadata_column
]
md, md_offset = tskit.pack_bytes(encoded_metadata_column)
mdR, md_offsetR = tskit.pack_bytes(encoded_metadata_column_Remapped)

tables.individuals.set_columns(flags=range(ts.num_individuals), metadata=md, metadata_offset=md_offset)
tablesRemapped.individuals.set_columns(flags=range(ts.num_individuals), metadata=mdR, metadata_offset=md_offsetR)
ts_meta = tables.tree_sequence()
tsRemapped_meta = tablesRemapped.tree_sequence()

In [31]:
colours = {"A": "#feb72d", "C": "#802694", "M": "#228c8d", "O": "#a8db34", "H": "none", "U": "none"}

In [32]:
sampleColour = [(ind, colours[ts_meta.individual(ind.id).metadata["pop"]]) for ind in ts_meta.individuals()]
sampleColourR = [(ind, colours[tsRemapped_meta.individual(ind.id).metadata["pop"]]) for ind in tsRemapped_meta.individuals()]

In [33]:
# Plot the tree sequence
styles = []
# Create a style for each population, programmatically (or just type the string by hand)
css_string =  ".node > .sym {fill: none} "
for n, colour in sampleColour:
    # target the symbols only (class "sym")
    s = f".n{n.id} > .sym " + "{" + f"fill: {colour}; width: 5px; height: 15px;" + "}"
    styles.append(s)
css_string += " ".join(styles)
print(f'CSS string applied:\n    "{css_string}"')

CSS string applied:
    ".node > .sym {fill: none} .n0 > .sym {fill: #802694; width: 5px; height: 15px;} .n1 > .sym {fill: #802694; width: 5px; height: 15px;} .n2 > .sym {fill: #802694; width: 5px; height: 15px;} .n3 > .sym {fill: #802694; width: 5px; height: 15px;} .n4 > .sym {fill: #802694; width: 5px; height: 15px;} .n5 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n6 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n7 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n8 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n9 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n10 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n11 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n12 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n13 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n14 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n15 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n16 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n17 > .sym {fill: #228c

In [34]:
# Plot the tree sequence
stylesR = []
# Create a style for each population, programmatically (or just type the string by hand)
css_stringR =  ".node > .sym {fill: none} "
for n, colour in sampleColourR:
    # target the symbols only (class "sym")
    sR = f".n{n.id} > .sym " + "{" + f"fill: {colour}; width: 5px; height: 15px;" + "}"
    stylesR.append(sR)
css_stringR += " ".join(stylesR)
print(f'CSS string applied:\n    "{css_stringR}"')

CSS string applied:
    ".node > .sym {fill: none} .n0 > .sym {fill: #802694; width: 5px; height: 15px;} .n1 > .sym {fill: #802694; width: 5px; height: 15px;} .n2 > .sym {fill: #802694; width: 5px; height: 15px;} .n3 > .sym {fill: #802694; width: 5px; height: 15px;} .n4 > .sym {fill: #802694; width: 5px; height: 15px;} .n5 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n6 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n7 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n8 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n9 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n10 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n11 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n12 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n13 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n14 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n15 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n16 > .sym {fill: #228c8d; width: 5px; height: 15px;} .n17 > .sym {fill: #228c

In [23]:
oneTree = ts_meta.at(100)
oneTreeRemapped = tsRemapped_meta.at(100)

In [24]:
wide_fmt = (1200, 500)
oneTree.draw_svg(
    size=wide_fmt,
    node_labels={},    # Remove all node labels for a clearer viz
    style=css_string
)

In [27]:
wide_fmt = (1200, 500)
oneTreeRemapped.draw_svg(
    size=wide_fmt,
    node_labels={},    # Remove all node labels for a clearer viz
    style=css_stringR
)

In [ ]:
#Plot the one tree and colour by lineage
styles = []
# Create a style for each population, programmatically (or just type the string by hand)
for colour, p in zip(['#feb72d', '#a8db34', '#228c8d', '#802694'], ts_full.populations()):
    # target the symbols only (class "sym")
    s = f".node.p{p.id} > .sym " + "{" + f"fill: {colour}" + "}"
    styles.append(s)
    print(f'"{s}" applies to nodes from population {p.metadata["name"]} (id {p.id})')
css_string = " ".join(styles)
print(f'CSS string applied:\n    "{css_string}"')
